# InvoiceAI — Quickstart

Extract structured data from invoices and receipts with **Qwen2.5-VL-3B**.

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Get the code

In [ ]:
GITHUB_USER = "YOUR_GITHUB_USERNAME"  # <- change this
REPO = "invoice-ai"

import os
if not os.path.exists(REPO):
    !git clone https://github.com/{GITHUB_USER}/{REPO}.git
%cd {REPO}
!git pull

## 3. Install requirements

In [ ]:
!pip install -q -r requirements.txt

## 4. Load the model

The first run downloads about 7 GB (takes a few minutes).

If the output is empty or looks like `!!!!!!`, float16 has a problem on this GPU.
Then run `os.environ["INVOICEAI_DTYPE"] = "bfloat16"` **before** this cell, restart the runtime (`Runtime → Restart session`) and run again.

In [ ]:
import json
import logging
import sys
import time

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", stream=sys.stdout)

from app.config import settings
from app.extractor import get_extractor

print(settings)
extractor = get_extractor()

start = time.time()
extractor.load()
print(f"Model loaded in {time.time() - start:.1f} s")

## 5. Run extraction on the sample images

In [ ]:
from pathlib import Path
from IPython.display import display
from PIL import Image

from app.extractor import ExtractionError

sample_files = sorted(
    p for p in Path("samples").iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)
print(f"Found {len(sample_files)} sample images")

for path in sample_files[:3]:
    print("=" * 80)
    print(path.name)
    preview = Image.open(path)
    preview.thumbnail((500, 500))
    display(preview)

    start = time.time()
    try:
        invoice = extractor.extract(path)
        print(json.dumps(invoice.model_dump(mode="json"), indent=2, ensure_ascii=False))
    except ExtractionError as exc:
        print(f"FAILED: {exc}")
        print("Raw model output:")
        print(extractor.last_raw_output)
    print(f"Time: {time.time() - start:.1f} s")

## 6. Try your own image (optional)

Upload a file with the folder icon on the left, then change the path below.

In [ ]:
invoice = extractor.extract("samples/your_file.jpg")  # <- change this
print(invoice.model_dump_json(indent=2))